# 04 · Final pipeline and explainable demonstration
**Question:** How does a new judgment become a reviewable brief?

**PDF/TXT → source sentences → trained scores and roles → evidence-grounded sections → audit → human review.**
The existing trained weights and checkpoint checks are preserved. Version 4.6 improves decision attribution, retains qualifications and recovers some unclosed PDF quotations with a review warning. See `docs/decision_upgrade.md` for development results and remaining errors; no new unseen-document accuracy is claimed.

## Setup — run once
The project uses an isolated Python environment so its Hugging Face packages do not conflict with Colab's installed Gradio/Diffusers.
`%%project` means “run this ordinary Python cell in that environment.” Variables persist between these cells.
The first setup downloads dependencies and can take several minutes. No token is needed for the demo or downloaded trained model.
For the gated role dataset, accept its access conditions and add `HF_TOKEN` in Colab Secrets; never paste it into a cell.

In [ ]:
from pathlib import Path
import os, subprocess, sys
ROOT = Path(os.environ.get('PROJECT_ROOT', '/content/legal-judgment-summarizer' if Path('/content').exists() else Path.cwd()))
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src').exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/Mananfrfr/legal-judgment-summarizer.git', str(ROOT)])
sys.path.insert(0, str(ROOT / 'src'))
from notebook_runtime import setup
setup(ROOT)


In [ ]:
%%project
from pathlib import Path
import os, json
import pandas as pd
from IPython.display import display, Markdown
ROOT = Path(os.environ['PROJECT_ROOT'])


## 1. Choose the demonstration mode
The invented TXT example runs without downloads. For a real trained-model demo, run `scripts/download_models.py` once, set `USE_TRAINED_MODEL=True`, and provide a judgment path. The supplied model remains your actual fine-tuned encoder and trained heads; we do not substitute a base BERT model.

In [ ]:
%%project
from project_api import load_deployed_model, summarize_document, save_summary
USE_TRAINED_MODEL = False
INPUT = ROOT / 'examples/demo_judgment.txt'
BUDGET = 800
model = load_deployed_model(ROOT) if USE_TRAINED_MODEL else None
print('TRAINED MODEL' if model else 'EVIDENCE-ONLY TEACHING DEMO: no trained ranking')


## 2. Inspect what was extracted
The PDF reader preserves page references and character spans. Quotations, earlier rulings and party submissions require attribution checks. OCR is optional and needs Tesseract; start with a searchable English PDF for your presentation.

In [ ]:
%%project
from stage44_source import read_document
document = read_document(INPUT, ocr='off')
display(pd.DataFrame(document.sentences)[['id', 'page', 'sentence']].head(10))
print('Extracted sentences:', len(document.sentences))


## 3. Why long judgments no longer stop at sentence 120
The trained selector originally used 120 sampled candidates. The current inference engine applies it to overlapping windows of 120 sentences, with overlap 30, then averages scores where windows overlap. This covers all extracted candidates, including the ending. It is a new inference regime and must be evaluated separately.

In [ ]:
%%project
from stage44_windows import window_ranges
print('Windows for a hypothetical 350-sentence judgment:')
print(list(window_ranges(350)))


## 4. Produce the brief
The model ranks candidates; source and attribution rules organize background, issues, reasoning and directions. Complete operative passages can exceed the requested word target. Exact-span checks reject modified source evidence. A missing section is shown explicitly rather than invented.

In [ ]:
%%project
result = summarize_document(INPUT, model=model, budget=BUDGET, ocr='off')
display(Markdown(result['markdown']))


## 5. Inspect the audit before trusting the result
Exact text matching checks copying. It cannot prove attribution, completeness or legal correctness. Compare the final pages manually, especially the outcome, amounts, recipients, deadlines, exceptions and negation.

In [ ]:
%%project
display(pd.DataFrame([{'mode': result['model']['mode'],
    'exact_source_check': result['audit']['exact_source_check_passed'],
    'missing_sections': ', '.join(result['audit']['missing_sections']),
    'flags': ', '.join(result['audit']['flags'])}]))
saved = save_summary(result, ROOT / 'outputs/presentation_demo')
print('Saved summary and evidence:', saved)


## 6. Use the browser app
On a computer with the dependencies and model installed, run `python src/stage44_app.py` from the repository. Open `http://127.0.0.1:7861` **on that same computer**. Colab's remote localhost is not your laptop's localhost.
The app shows source pages, exact highlights and separately saved reviewer drafts. It is a local research preview; the Colab batch notebook is the supported cloud workflow.

### A two-minute live demo
1. Upload a short searchable judgment.
2. Generate the brief and open an evidence reference.
3. Jump to the final pages and check the decision.
4. Explain a warning, correct the reviewer draft and export it.

**Next research step:** obtain independent paragraph/decision annotations, improve attribution and calibrated abstention, and evaluate on a new untouched set. Do not promise that the existing three failure cases prove performance after fixing them.